# Lesson 6.7: The Paired t-Test

Test whether a drug lowers blood pressure in the lesson's 8 patients measured before and after, with a paired t-test.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Enter the paired data

Each patient appears in both lists, in the same order, so each 'after' belongs to a particular 'before'. The difference for each patient is their own change, and the lesson says every single one improved.

In [1]:
import numpy as np
from scipy import stats

before = np.array([142, 138, 155, 129, 147, 161, 133, 145])
after = np.array([135, 134, 148, 126, 139, 152, 130, 137])
diff = before - after

print('differences:', diff)
print(f'mean difference = {diff.mean():.3f}')
print(f'sd of differences = {diff.std(ddof=1):.3f}')

differences: [7 4 7 3 8 9 3 8]
mean difference = 6.125
sd of differences = 2.416


## 2. Run the paired t-test

ttest_rel compares before and after pair by pair. A positive t means the 'before' values were higher. We also add a 95% confidence interval for the average drop.

In [2]:
paired = stats.ttest_rel(before, after)
ci = paired.confidence_interval(confidence_level=0.95)

print(f't  = {paired.statistic:.3f}')
print(f'df = {paired.df}')
print(f'p  = {paired.pvalue:.4f}')
print(f'95% CI for the drop: ({ci.low:.2f}, {ci.high:.2f})')

t  = 7.169
df = 7
p  = 0.0002
95% CI for the drop: (4.10, 8.15)


## 3. Same result as a one-sample test

A paired t-test is just a one-sample t-test on the differences, testing whether their mean is 0. The numbers come out exactly the same.

In [3]:
one_sample = stats.ttest_1samp(diff, popmean=0)

print(f't  = {one_sample.statistic:.3f}')
print(f'p  = {one_sample.pvalue:.4f}')
print('same t as the paired test?', np.isclose(one_sample.statistic, paired.statistic))

t  = 7.169
p  = 0.0002
same t as the paired test? True


## 4. What the wrong test says

If we ignore the pairing and treat the 16 numbers as two independent groups, the big differences between patients swamp the 6 mmHg drop. The result is far less convincing.

In [4]:
wrong = stats.ttest_ind(before, after, equal_var=False)

print(f'independent test: t = {wrong.statistic:.2f}, p = {wrong.pvalue:.3f}')
print(f'paired test:      t = {paired.statistic:.2f}, p = {paired.pvalue:.4f}')

independent test: t = 1.25, p = 0.231
paired test:      t = 7.17, p = 0.0002


## Try this

- Change the last 'after' value from 137 to 145 and predict what happens to t and p, then run it.
- Shuffle the order of the 'after' list (break the pairing) and run the paired test again. What happens?

## The same idea in R, SPSS and Stata

**R**

```r
before <- c(142, 138, 155, 129, 147, 161, 133, 145)
after  <- c(135, 134, 148, 126, 139, 152, 130, 137)

t.test(before, after, paired = TRUE)
t.test(before - after, mu = 0)    # same thing: one-sample test on the differences
```

**SPSS** (Analyze > Compare Means > Paired-Samples T Test)

```
DATA LIST FREE / before after.
BEGIN DATA
142 135 138 134 155 148 129 126
147 139 161 152 133 130 145 137
END DATA.
T-TEST PAIRS=before WITH after (PAIRED)
  /CRITERIA=CI(.95).
```

**Stata**

```
clear
input before after
142 135
138 134
155 148
129 126
147 139
161 152
133 130
145 137
end
ttest before == after
```